# Stress Prediction — 이상치 Winsorize (노이즈 큰 피처 다듬기)

EDA에서 이상치 개수: mean_working(33개, 1.10%) > diastolic_blood_pressure(19개) >
cholesterol(17개) > glucose(13개) > weight(2) > systolic_blood_pressure(2) > height(1).

다른 팀 힌트("mean_working 직접 넣으면 불안정")와 일치합니다. **제거가 아니라
IQR 기준으로 극단값만 눌러주는(winsorize) 방식**을 테스트합니다 — 정보를 아예
버리지 않으면서 노이즈만 줄이는 방향이라, 제거보다 정보 손실이 적습니다.

**train의 IQR 경계로만 계산**하고(leakage 방지), test/val은 그 경계로 clip합니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 800  # 탐색용
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]

# EDA에서 이상치가 많았던 순서
WINSORIZE_CANDIDATES = ["mean_working", "diastolic_blood_pressure", "cholesterol", "glucose"]


def compute_iqr_bounds(train_frame, columns):
    bounds = {}
    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame, bounds):
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


## CV 비교: winsorize 없음 vs 있음 (누적 1개 -> 4개)

In [2]:

train_raw = pd.read_csv(DATA_DIR / "train.csv")
y = train_raw[TARGET].reset_index(drop=True)
raw_input = train_raw.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)


def cv_mae_with_winsorize(winsorize_cols):
    oof = np.zeros(len(y))
    for train_idx, val_idx in cv.split(raw_input):
        train_raw_fold = raw_input.iloc[train_idx].reset_index(drop=True)
        val_raw_fold = raw_input.iloc[val_idx].reset_index(drop=True)
        y_train = y.iloc[train_idx].reset_index(drop=True)

        if winsorize_cols:
            bounds = compute_iqr_bounds(train_raw_fold, winsorize_cols)
            train_raw_fold = apply_winsorize(train_raw_fold, bounds)
            val_raw_fold = apply_winsorize(val_raw_fold, bounds)

        train_feat = add_features(train_raw_fold)
        val_feat = add_features(val_raw_fold)

        pred_tree = tree_prediction(train_feat, val_feat, y_train)
        pred_pair = pair_prediction(train_feat, val_feat, y_train)
        combined = np.clip(np.round((1 - PAIR_WEIGHT) * pred_tree + PAIR_WEIGHT * pred_pair, 2), 0.0, 1.0)
        oof[val_idx] = combined
    return mean_absolute_error(y, oof)


baseline_mae = cv_mae_with_winsorize([])
print(f"### baseline(winsorize 없음, gender 제거) CV MAE: {baseline_mae:.6f}")
print()

rows = [{"n_winsorized": 0, "columns": "(none)", "cv_mae": baseline_mae}]
for n in range(1, len(WINSORIZE_CANDIDATES) + 1):
    cols = WINSORIZE_CANDIDATES[:n]
    mae = cv_mae_with_winsorize(cols)
    rows.append({"n_winsorized": n, "columns": ", ".join(cols), "cv_mae": mae})
    print(f"상위 {n}개 winsorize {cols}: {mae:.6f} (delta {mae - baseline_mae:+.6f})")

result_df = pd.DataFrame(rows)
print()
display(result_df)
print()
print("(참고) v34(gender 제거) 오늘 CV: 0.147650~0.147787")


### baseline(winsorize 없음, gender 제거) CV MAE: 0.147697

상위 1개 winsorize ['mean_working']: 0.147477 (delta -0.000220)
상위 2개 winsorize ['mean_working', 'diastolic_blood_pressure']: 0.147643 (delta -0.000053)
상위 3개 winsorize ['mean_working', 'diastolic_blood_pressure', 'cholesterol']: 0.147993 (delta +0.000297)
상위 4개 winsorize ['mean_working', 'diastolic_blood_pressure', 'cholesterol', 'glucose']: 0.147390 (delta -0.000307)



,n_winsorized,columns,cv_mae
0,0,(none),0.147697
1,1,mean_working,0.147477
2,2,"mean_working, diastolic_blood_pressure",0.147643
3,3,"mean_working, diastolic_blood_pressure, choles...",0.147993
4,4,"mean_working, diastolic_blood_pressure, choles...",0.147390



(참고) v34(gender 제거) 오늘 CV: 0.147650~0.147787


## 최종 제출 파일 생성

In [3]:

BEST_N_WINSORIZED = 0  # 결과 1위로 수정

test_raw = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_full_input = raw_input
test_full_input = test_raw.drop(columns=[ID_COLUMN]).reset_index(drop=True)

winsorize_cols = WINSORIZE_CANDIDATES[:BEST_N_WINSORIZED]
if winsorize_cols:
    bounds = compute_iqr_bounds(train_full_input, winsorize_cols)
    train_full_input = apply_winsorize(train_full_input, bounds)
    test_full_input = apply_winsorize(test_full_input, bounds)

final_train_feat = add_features(train_full_input)
final_test_feat = add_features(test_full_input)

N_ESTIMATORS = 1200
final_tree = tree_prediction(final_train_feat, final_test_feat, y)
final_pair = pair_prediction(final_train_feat, final_test_feat, y)
final_prediction = np.clip(np.round((1 - PAIR_WEIGHT) * final_tree + PAIR_WEIGHT * final_pair, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_winsorize_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_winsorize_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.46
8,TEST_0008,0.57
9,TEST_0009,0.88
